# Análise do tamanho das redes neurais

Este notebook mede o tamanho de cada componente do pipeline de difusão condicional:

| Componente | Papel |
|---|---|
| `UNet_cond` | Rede de difusão no espaço latente (cross-attention com identidade) |
| `VAE` | Encoder/decoder imagem ↔ latente 4×32×32 |
| `ArcFaceOnlyEncoder` | Projeta embedding ArcFace (512-d) em 16 tokens de contexto |
| `CLIPAttributeClassifier` | Classificador dos 40 atributos CelebA (CLIP ViT-L/14 + head) |

Para cada um: parâmetros totais/treináveis, memória em FP32/FP16, breakdown por submódulo
e, no final, um resumo comparativo com gráfico e o tamanho dos checkpoints em disco.

> Roda inteiramente em CPU — nenhum forward pass é necessário para contar parâmetros.

In [ ]:
import os, sys, glob
from collections import OrderedDict

import torch
import torch.nn as nn
import pandas as pd
import matplotlib.pyplot as plt

# garante import a partir da raiz do projeto
PROJECT_ROOT = os.path.abspath(".")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

torch.set_grad_enabled(False)
pd.options.display.float_format = "{:,.2f}".format
print(f"PyTorch {torch.__version__} | raiz: {PROJECT_ROOT}")

## Funções auxiliares

- `param_stats` — parâmetros totais/treináveis e memória (parâmetros + buffers);
- `module_breakdown` — tabela por submódulo de primeiro nível (ou profundidade maior);
- `fmt_count` — formata `1234567 → 1.23 M`.

In [ ]:
def fmt_count(n):
    if n >= 1e9:  return f"{n/1e9:.2f} B"
    if n >= 1e6:  return f"{n/1e6:.2f} M"
    if n >= 1e3:  return f"{n/1e3:.1f} K"
    return str(n)

def fmt_bytes(b):
    for unit in ("B", "KB", "MB", "GB"):
        if b < 1024:
            return f"{b:.2f} {unit}"
        b /= 1024
    return f"{b:.2f} TB"

def param_stats(model, name):
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    buffers   = sum(b.numel() for b in model.buffers())
    bytes_fp32 = (total + buffers) * 4
    return {
        "componente":  name,
        "params":      total,
        "treináveis":  trainable,
        "congelados":  total - trainable,
        "buffers":     buffers,
        "FP32":        fmt_bytes(bytes_fp32),
        "FP16":        fmt_bytes(bytes_fp32 / 2),
    }

def show_stats(model, name):
    s = param_stats(model, name)
    print(f"{name}")
    print(f"  parâmetros : {fmt_count(s['params'])}  ({s['params']:,})")
    print(f"  treináveis : {fmt_count(s['treináveis'])}  "
          f"({100*s['treináveis']/max(s['params'],1):.1f}%)")
    print(f"  memória    : {s['FP32']} (FP32)  |  {s['FP16']} (FP16)")
    return s

def module_breakdown(model, depth=1):
    rows = []
    for name, mod in model.named_modules():
        if name == "" or name.count(".") != depth - 1:
            continue
        n = sum(p.numel() for p in mod.parameters())
        if n == 0:
            continue
        rows.append({"módulo": name,
                     "params": n,
                     "params_fmt": fmt_count(n),
                     "% do total": 100 * n / sum(p.numel() for p in model.parameters())})
    df = pd.DataFrame(rows).sort_values("params", ascending=False).reset_index(drop=True)
    return df

## 1. U-Net condicional (`UNet_cond`)

Configuração usada no treino/geração (`generate_cfg_composable.py`):
`in_channels=4, out_channels=4, base_channels=128, channel_mults=(1,2,4), context_dim=512`.

In [ ]:
from models.unet_conditional import UNet_cond

unet = UNet_cond(in_channels=4, out_channels=4, context_dim=512)
unet_stats = show_stats(unet, "UNet_cond")

In [ ]:
# breakdown por bloco de primeiro nível (encoder / mid / decoder / embeddings)
module_breakdown(unet, depth=1)

### Onde estão os parâmetros da U-Net?

Separando por tipo de camada — quanto vai para os `ResidualBlock` (convoluções)
versus os `SpatialTransformer` (cross-attention, onde entra o condicionamento de identidade).

In [ ]:
from models.modules import ResidualBlock, SpatialTransformer

by_type = {}
for mod in unet.modules():
    t = type(mod).__name__
    if isinstance(mod, (ResidualBlock, SpatialTransformer)):
        # conta só parâmetros diretos + filhos que não são desses tipos,
        # para não somar duas vezes módulos aninhados
        n = sum(p.numel() for p in mod.parameters())
        by_type.setdefault(t, 0)
        by_type[t] += n

# ResidualBlocks dentro de SpatialTransformer não existem nesta arquitetura,
# então a soma direta não duplica nada.
total = unet_stats["params"]
outros = total - sum(by_type.values())
by_type["outros (conv_in/out, time_mlp, up/downsample)"] = outros

pd.DataFrame(
    [{"tipo": k, "params": fmt_count(v), "% do total": f"{100*v/total:.1f}%"}
     for k, v in sorted(by_type.items(), key=lambda kv: -kv[1])]
)

## 2. VAE

`VAE(in_channels=3, latent_dim=4)` — congelado durante o treino da difusão,
mas precisa estar em memória na geração.

In [ ]:
from vae.modules import VAE

vae = VAE(in_channels=3, latent_dim=4)
vae_stats = show_stats(vae, "VAE")
module_breakdown(vae, depth=1)

## 3. Encoder de identidade

O treino paired está sendo feito com `--encoder clip_arcface`, ou seja, o
`ImageConditionEncoder` (estilo IP-Adapter): CLIP ViT-B/32 **congelado** + projeções
treináveis (`clip_proj` e `id_proj`). Mude `ENCODER_TYPE` para `"arcface_only"` se
quiser analisar a variante sem CLIP.

O backbone ArcFace é um modelo ONNX externo (w600k_r50, ~166 MB em disco) e não aparece
em `parameters()` do PyTorch em nenhuma das variantes.

⚠️ O `ImageConditionEncoder` baixa o CLIP ViT-B/32 do HuggingFace (~600 MB) na
primeira execução.

In [ ]:
from models.encoders import ArcFaceOnlyEncoder, ImageConditionEncoder

ENCODER_TYPE = "clip_arcface"   # ou "arcface_only"

if ENCODER_TYPE == "clip_arcface":
    id_encoder = ImageConditionEncoder(context_dim=512, num_tokens=16, freeze_backbone=True)
    id_stats = show_stats(id_encoder, "ImageConditionEncoder (CLIP + ArcFace)")
    clip_frozen = sum(p.numel() for p in id_encoder.clip.parameters())
    clip_proj_n = sum(p.numel() for p in id_encoder.clip_proj.parameters())
    id_proj_n   = sum(p.numel() for p in id_encoder.id_proj.parameters()) \
                + sum(p.numel() for p in id_encoder.id_norm.parameters())
    print()
    print(f"  CLIP ViT-B/32 (congelado)      : {fmt_count(clip_frozen)}")
    print(f"  clip_proj (treinável)          : {fmt_count(clip_proj_n)}")
    print(f"  id_proj + id_norm (treinável)  : {fmt_count(id_proj_n)}")
else:
    id_encoder = ArcFaceOnlyEncoder(context_dim=512, num_tokens=16)
    id_stats = show_stats(id_encoder, "ArcFaceOnlyEncoder (projeção treinável)")

display(module_breakdown(id_encoder, depth=1))

# tamanho do ONNX do ArcFace em disco, se já baixado
arcface_paths = glob.glob(os.path.expanduser("~/.insightface/models/buffalo_l/w600k_r50.onnx"))
if arcface_paths:
    print(f"\nArcFace ONNX (congelado): {fmt_bytes(os.path.getsize(arcface_paths[0]))} em disco")
else:
    print("\nArcFace ONNX ainda não baixado nesta máquina (~166 MB quando baixado).")

## 4. Classificador de atributos (`CLIPAttributeClassifier`)

⚠️ Esta célula carrega o CLIP ViT-L/14 do HuggingFace (~1.2 GB no primeiro download).
Se não quiser baixar agora, defina `ANALYZE_CLIP = False` — o resto do notebook continua funcionando.

In [ ]:
ANALYZE_CLIP = True

clf_stats = None
if ANALYZE_CLIP:
    try:
        from models.attribute_classifier import CLIPAttributeClassifier
        clf = CLIPAttributeClassifier(num_attributes=40, unfreeze_last_n=4)
        clf_stats = show_stats(clf, "CLIPAttributeClassifier")
        print()
        head_params = sum(p.numel() for p in clf.head.parameters())
        clip_params = sum(p.numel() for p in clf.clip.parameters())
        clip_train  = sum(p.numel() for p in clf.clip.parameters() if p.requires_grad)
        print(f"  backbone CLIP : {fmt_count(clip_params)} "
              f"(treináveis nos últimos 4 blocos: {fmt_count(clip_train)})")
        print(f"  head (40 attr): {fmt_count(head_params)}")
    except Exception as e:
        print(f"CLIP indisponível ({type(e).__name__}: {e})")
        print("Estimativa analítica: ViT-L/14 vision ≈ 303 M params + head ≈ 0.55 M.")
else:
    print("Análise do CLIP desativada (ANALYZE_CLIP = False).")

## 5. Resumo comparativo

In [ ]:
all_stats = [unet_stats, vae_stats, id_stats]
if clf_stats is not None:
    all_stats.append(clf_stats)

summary = pd.DataFrame(all_stats)
summary["params_fmt"]     = summary["params"].map(fmt_count)
summary["treináveis_fmt"] = summary["treináveis"].map(fmt_count)

total_row = {
    "componente": "TOTAL (pipeline de geração)",
    "params": summary["params"].sum(),
    "treináveis": summary["treináveis"].sum(),
}
print(f"Total do pipeline : {fmt_count(total_row['params'])} parâmetros")
print(f"Total treinável   : {fmt_count(total_row['treináveis'])}")
print(f"Memória FP32      : {fmt_bytes(total_row['params']*4)}"
      f"  |  FP16: {fmt_bytes(total_row['params']*2)}")

summary[["componente", "params_fmt", "treináveis_fmt", "FP32", "FP16"]]

In [ ]:
# gráfico: parâmetros por componente, divididos em treináveis × congelados
plt.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.spines.top": False, "axes.spines.right": False,
    "grid.color": "#e6e6e6", "axes.axisbelow": True, "font.size": 11,
})

df = summary.sort_values("params")
labels    = df["componente"]
trainable = df["treináveis"] / 1e6
frozen    = df["congelados"] / 1e6

fig, ax = plt.subplots(figsize=(9, 0.7 * len(df) + 1.5))
ax.barh(labels, trainable, height=0.55, color="#2a78d6", label="Treináveis")
ax.barh(labels, frozen, left=trainable, height=0.55, color="#b8c4d0", label="Congelados")

for i, (t, f) in enumerate(zip(trainable, frozen)):
    ax.text(t + f + max(trainable + frozen) * 0.01, i,
            fmt_count((t + f) * 1e6), va="center", color="#444")

ax.set_xlabel("Parâmetros (milhões)")
ax.set_title("Tamanho das redes por componente", loc="left", fontweight="bold")
ax.legend(frameon=False, loc="lower right")
ax.grid(axis="x")
plt.tight_layout()
plt.show()

## 6. O que é treinado de fato

`requires_grad` na instanciação não conta a história toda — o que importa é o que cada
script de treino passa ao otimizador:

- **Difusão** (`train_cfg_composable[_paired].py`): o `AdamW` recebe
  `UNet + image_encoder + attribute_embedder`. Dentro do `ImageConditionEncoder`
  o backbone CLIP ViT-B/32 está com `requires_grad=False`, então só as projeções
  recebem gradiente. O VAE é congelado e o ArcFace é ONNX, fora do grafo.
- **Classificador** (`train_attribute_classifier.py`): só os últimos 4 blocos do
  CLIP ViT-L/14 + head — o resto do backbone fica congelado.

Os dois treinos são independentes, então cada linha abaixo é o "orçamento" de gradientes
daquele treino (é isso que dimensiona a memória do otimizador: AdamW guarda 2 estados
FP32 por parâmetro treinado, ou seja, +8 bytes/param além do peso).

In [ ]:
from models.modules import AttributeEmbedder

attr_embedder = AttributeEmbedder(num_attributes=40, context_dim=512)

def n_params(m, only_trainable=False):
    return sum(p.numel() for p in m.parameters()
               if not only_trainable or p.requires_grad)

# --- treino da difusão -------------------------------------------------
enc_name = type(id_encoder).__name__
diff_parts = {
    "UNet_cond":                     n_params(unet, only_trainable=True),
    f"{enc_name} (projeções)":       n_params(id_encoder, only_trainable=True),
    "AttributeEmbedder":             n_params(attr_embedder, only_trainable=True),
}
diff_total = sum(diff_parts.values())
enc_frozen = n_params(id_encoder) - n_params(id_encoder, only_trainable=True)

print(f"Treino da difusão (train_cfg_composable_paired.py, encoder={ENCODER_TYPE}):")
for k, v in diff_parts.items():
    print(f"  {k:32s} {fmt_count(v):>10s}")
print(f"  {'TOTAL treinado':32s} {fmt_count(diff_total):>10s}")
print(f"  estado do AdamW (2× FP32): {fmt_bytes(diff_total * 8)}")
frozen_msg = f"VAE ({fmt_count(n_params(vae))}) + ArcFace ONNX"
if enc_frozen:
    frozen_msg += f" + CLIP ViT-B/32 ({fmt_count(enc_frozen)})"
print(f"  congelados neste treino  : {frozen_msg}")

# --- treino do classificador -------------------------------------------
print("\nTreino do classificador (train_attribute_classifier.py):")
if clf_stats is not None:
    clf_train = n_params(clf, only_trainable=True)
    clf_total = n_params(clf)
    print(f"  treináveis (últimos 4 blocos + head): {fmt_count(clf_train)}"
          f" de {fmt_count(clf_total)} ({100*clf_train/clf_total:.1f}%)")
    print(f"  estado do AdamW (2× FP32): {fmt_bytes(clf_train * 8)}")
else:
    print("  (CLIP não carregado) estimativa analítica para ViT-L/14:")
    print("  4 blocos × 12.60 M + head 0.55 M ≈ 50.93 M treináveis de ~303 M (16.8%)")

## 7. Checkpoints em disco

Tamanho real dos arquivos `.pt` / `.pth` / `.ckpt` no projeto — inclui otimizador e EMA
quando salvos juntos, por isso pode ser bem maior que o modelo sozinho.

In [ ]:
ckpts = sorted(
    glob.glob(os.path.join(PROJECT_ROOT, "**", "*.pt"), recursive=True)
    + glob.glob(os.path.join(PROJECT_ROOT, "**", "*.pth"), recursive=True)
    + glob.glob(os.path.join(PROJECT_ROOT, "**", "*.ckpt"), recursive=True)
)
rows = []
for path in ckpts:
    size = os.path.getsize(path)
    rows.append({"arquivo": os.path.relpath(path, PROJECT_ROOT), "tamanho": fmt_bytes(size)})

if rows:
    display(pd.DataFrame(rows))
else:
    print("Nenhum checkpoint local — os pesos ficam na VM de deploy.")

In [ ]:
# inspeciona o conteúdo de um checkpoint (chaves e tamanho de cada state_dict)
CKPT_PATH = os.path.join(PROJECT_ROOT, "models", "ckpt.pt")

if os.path.exists(CKPT_PATH):
    ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
    if isinstance(ckpt, dict):
        for key, val in ckpt.items():
            if isinstance(val, dict) and val and all(torch.is_tensor(v) for v in val.values()):
                n = sum(v.numel() for v in val.values())
                b = sum(v.numel() * v.element_size() for v in val.values())
                print(f"  {key:35s} {fmt_count(n):>10s} params  ({fmt_bytes(b)})")
            else:
                print(f"  {key:35s} {type(val).__name__}")
else:
    print(f"{CKPT_PATH} não encontrado — ajuste CKPT_PATH para outro checkpoint.")